# Stažení dat

**Zdroj:** [Jolpica F1 API](https://github.com/jolpica/jolpica-f1). Vrací data ve formátu **JSON**, který převedeme na tabulky **CSV**.

Stahujeme sezóny **2014–2026** (hybridní éra a nová pravidla 2026):
- `races` – kalendář (sezóna, kolo, trať, datum)
- `results` – výsledky závodů (pozice, start z gridu, body, status)
- `qualifying` – kvalifikace (Q1/Q2/Q3)

Odpovědi se ukládají do `data/raw/json/`. Při opětovném spuštění se nic nestahuje znovu, kromě rozjeté sezóny, pokud nastavíš `REFRESH_CURRENT = True`.

Stahujeme data 17.10 abychom mohli porovnat jak model bude přesný v rámci zbývajících 5 zavodů

In [1]:
import json, time, hashlib
from pathlib import Path
import requests
import pandas as pd

BASE = "https://api.jolpi.ca/ergast/f1"
SEASONS = range(2014, 2027)
CURRENT_SEASON = 2026
REFRESH_CURRENT = False      # True = znovu stáhnout rozjetou sezónu (nové závody) Při prvním spuštění se automaticky stáhnout data

DATA = Path("../data")
CACHE = DATA / "raw" / "json"
CACHE.mkdir(parents=True, exist_ok=True)

In [2]:
def fetch(path, params=None, refresh=False):
    """GET s cache na disku a opakováním při HTTP 429 (rate limit)."""
    params = params or {}
    key = path.strip("/").replace("/", "_") + "_" + "_".join(f"{k}{v}" for k, v in sorted(params.items()))
    f = CACHE / f"{key}.json"
    if f.exists() and not refresh:
        return json.loads(f.read_text())
    for attempt in range(6):
        r = requests.get(f"{BASE}/{path}", params=params, timeout=30)
        if r.status_code == 429:
            time.sleep(5 * (attempt + 1)); continue
        r.raise_for_status()
        data = r.json()
        f.write_text(json.dumps(data))
        time.sleep(0.3)
        return data
    raise RuntimeError(f"Nepodařilo se stáhnout {path}")


def fetch_all(path, refresh=False, limit=100):
    """Stránkování přes limit/offset – vrací seznam závodů (Races) ze všech stránek."""
    races, offset = [], 0
    while True:
        d = fetch(path, {"limit": limit, "offset": offset}, refresh)["MRData"]
        races += d["RaceTable"]["Races"]
        offset += limit
        if offset >= int(d["total"]):
            return races

## Stažení a převod JSON → tabulky

In [3]:
race_rows, result_rows, quali_rows = [], [], []

for season in SEASONS:
    refresh = REFRESH_CURRENT and season == CURRENT_SEASON
    for r in fetch(f"{season}.json", {"limit": 100}, refresh)["MRData"]["RaceTable"]["Races"]:
        race_rows.append({
            "season": int(r["season"]), "round": int(r["round"]), "race_name": r["raceName"],
            "date": r["date"], "circuit_id": r["Circuit"]["circuitId"],
            "circuit_name": r["Circuit"]["circuitName"], "country": r["Circuit"]["Location"]["country"],
        })
    for r in fetch_all(f"{season}/results.json", refresh):
        for x in r["Results"]:
            result_rows.append({
                "season": int(r["season"]), "round": int(r["round"]),
                "driver_id": x["Driver"]["driverId"], "driver_code": x["Driver"].get("code"),
                "driver_nationality": x["Driver"].get("nationality"), "driver_dob": x["Driver"].get("dateOfBirth"),
                "constructor_id": x["Constructor"]["constructorId"],
                "grid": int(x["grid"]), "position": int(x["position"]), "position_text": x["positionText"],
                "points": float(x["points"]), "laps": int(x["laps"]), "status": x["status"],
                "time_millis": int(x["Time"]["millis"]) if "Time" in x else None,
            })
    for r in fetch_all(f"{season}/qualifying.json", refresh):
        for x in r["QualifyingResults"]:
            quali_rows.append({
                "season": int(r["season"]), "round": int(r["round"]),
                "driver_id": x["Driver"]["driverId"], "quali_position": int(x["position"]),
                "q1": x.get("Q1"), "q2": x.get("Q2"), "q3": x.get("Q3"),
            })
    print(f"{season}: hotovo")

races = pd.DataFrame(race_rows)
results = pd.DataFrame(result_rows)
qualifying = pd.DataFrame(quali_rows)

2014: hotovo
2015: hotovo
2016: hotovo
2017: hotovo
2018: hotovo
2019: hotovo
2020: hotovo
2021: hotovo
2022: hotovo
2023: hotovo
2024: hotovo
2025: hotovo
2026: hotovo


In [4]:
for name, df in {"races": races, "results": results, "qualifying": qualifying}.items():
    df.to_csv(DATA / f"{name}.csv", index=False)
    print(f"{name:11s} {df.shape}")
results.tail()

races       (275, 7)
results     (5479, 14)
qualifying  (5458, 7)


,season,round,driver_id,driver_code,driver_nationality,driver_dob,constructor_id,grid,position,position_text,points,laps,status,time_millis
5474,2026,17,albon,ALB,Thai,1996-03-23,williams,16,18,18,0.0,59,Retired,5998746.0
5475,2026,17,bottas,BOT,Finnish,1989-08-28,cadillac,18,19,19,0.0,59,Lapped,6138132.0
5476,2026,17,stroll,STR,Canadian,1998-10-29,aston_martin,21,20,R,0.0,36,Retired,NaN
5477,2026,17,colapinto,COL,Argentine,2003-05-27,alpine,15,21,R,0.0,29,Retired,NaN
5478,2026,17,bortoleto,BOR,Brazilian,2004-10-14,audi,14,22,R,0.0,7,Retired,NaN
